# 01. Intro to Graph RAG

> *Module 01 asked "what are the five vectors closest to this one?". This module asks a different question: "what's connected to this one, and through which path?". When the answer depends on relationships, not similarity, a graph database is the right tool.*

Three notebooks. **This is the crawl.** A tiny demo graph (three people, three movies, a handful of relationships) in a fresh Kuzu database, one schema definition, one MATCH query. No LLM yet, no LangChain yet. Just Cypher and the smallest possible Kuzu surface.

## What's a graph, exactly?

A graph database stores three things: **nodes**, **relationships**, and **properties** on either of them.

- A **node** is a thing. A person, a movie, a studio. It has a label (`Person`) and some properties (`name`, `born_year`).
- A **relationship** connects two nodes in a specific direction. Keanu Reeves `ACTED_IN` The Matrix. The direction matters: the inverse isn't true. Properties can hang off relationships too, but we won't use that today.

That's it. Everything else (queries, traversals, aggregations) is built on those three primitives.

## Why this is a different shape of retrieval

Module 01's vector retriever embedded your query and asked the index for the closest points. Great for "find me docs about X". Useless for "find me the directors of films my favourite actor has been in".

A graph database answers questions where the answer is **a path through the data**: actor to film to director, or person to project to teammate to skill. You can do similarity on top of a graph too, but the headline operation is structural: follow the edges, return the nodes that match the pattern.

## Setup

Only NB3 needs OpenAI. NB1 and NB2 only need Kuzu, which is `pip install kuzu`. Kuzu is embedded: it runs in-process, stores the database as a single file on disk, no server, no Docker, no signup.

This notebook creates its own throwaway database at `data/intro_demo.kuzu` so it doesn't conflict with the bigger movie graph NB2 and NB3 use.

In [1]:
from helpers import get_kuzu_conn
from pathlib import Path

DB_PATH = Path("data") / "intro_demo.kuzu"

# Wipe any prior run so the schema CREATE statements below succeed.
# Kuzu 0.11+ stores the DB as a single file plus a sibling .wal write-ahead
# log. We need to delete both, otherwise re-opening replays the old schema.
# Older versions used a directory, hence the rmtree fallback.
for path in [DB_PATH, DB_PATH.with_suffix(DB_PATH.suffix + ".wal")]:
    if path.exists():
        if path.is_file():
            path.unlink()
        else:
            import shutil
            shutil.rmtree(path)

conn = get_kuzu_conn(DB_PATH)
print(f"Fresh Kuzu DB at {DB_PATH}")

Fresh Kuzu DB at data\intro_demo.kuzu


## Schema: the rules of the graph

Kuzu requires you to declare node and relationship types before you write any data. (This is closer to a SQL schema than to Neo4j's schemaless-by-default approach. It's actually a feature for graph RAG: an LLM writing Cypher from natural language needs a schema to ground itself in.)

We create two node tables and one relationship table:

In [2]:
conn.execute("""
    CREATE NODE TABLE Person(
        name STRING,
        PRIMARY KEY (name)
    )
""")

conn.execute("""
    CREATE NODE TABLE Movie(
        title STRING,
        year INT64,
        PRIMARY KEY (title)
    )
""")

conn.execute("CREATE REL TABLE ACTED_IN(FROM Person TO Movie)")

print("Schema created.")

Schema created.


## Insert a tiny dataset

Three people, three movies, six `ACTED_IN` edges. By hand.

In [3]:
for name in ["Keanu Reeves", "Carrie-Anne Moss", "Laurence Fishburne"]:
    conn.execute("CREATE (:Person {name: $name})", {"name": name})

for title, year in [("The Matrix", 1999), ("The Matrix Reloaded", 2003), ("John Wick", 2014)]:
    conn.execute(
        "CREATE (:Movie {title: $title, year: $year})",
        {"title": title, "year": year},
    )

edges = [
    ("Keanu Reeves", "The Matrix"),
    ("Keanu Reeves", "The Matrix Reloaded"),
    ("Keanu Reeves", "John Wick"),
    ("Carrie-Anne Moss", "The Matrix"),
    ("Carrie-Anne Moss", "The Matrix Reloaded"),
    ("Laurence Fishburne", "The Matrix"),
]
for actor, movie in edges:
    conn.execute(
        """
        MATCH (p:Person {name: $actor}), (m:Movie {title: $movie})
        CREATE (p)-[:ACTED_IN]->(m)
        """,
        {"actor": actor, "movie": movie},
    )

print("Inserted 3 people, 3 movies, 6 ACTED_IN edges.")

Inserted 3 people, 3 movies, 6 ACTED_IN edges.


## Your first MATCH

`MATCH` is the core retrieval clause. You draw a pattern, Kuzu finds every place in the graph that fits it, and you `RETURN` whatever you want.

Here's the pattern: a `Person` who `ACTED_IN` a `Movie`. The arrow direction matters.

In [4]:
conn.execute("""
    MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
    RETURN p.name AS actor, m.title AS movie, m.year AS year
    ORDER BY actor, year
""").get_as_df()

,actor,movie,year
0,Carrie-Anne Moss,The Matrix,1999
1,Carrie-Anne Moss,The Matrix Reloaded,2003
2,Keanu Reeves,The Matrix,1999
3,Keanu Reeves,The Matrix Reloaded,2003
4,Keanu Reeves,John Wick,2014
5,Laurence Fishburne,The Matrix,1999


## Walking two hops: co-actors

Now the move that vector search can't make. "Who acted in the same film as Keanu Reeves?"

In Cypher this is a single pattern with two relationships sharing a node. `(a)-[:ACTED_IN]->(m)<-[:ACTED_IN]-(b)`: actor `a` and actor `b` both pointing at the same movie `m`.

In [5]:
conn.execute("""
    MATCH (keanu:Person {name: 'Keanu Reeves'})-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(other:Person)
    RETURN other.name AS co_actor, m.title AS in_movie
    ORDER BY co_actor, in_movie
""").get_as_df()

,co_actor,in_movie
0,Carrie-Anne Moss,The Matrix
1,Carrie-Anne Moss,The Matrix Reloaded
2,Keanu Reeves,John Wick
3,Keanu Reeves,The Matrix
4,Keanu Reeves,The Matrix Reloaded
5,Laurence Fishburne,The Matrix


## What just happened

Four things to notice:

1. **The pattern is the query.** We didn't write a join, we drew a shape. The optimizer figured out the rest.
2. **Direction matters but pattern direction can be either way.** `(a)-[:ACTED_IN]->(m)<-[:ACTED_IN]-(b)` and `(a)-[:ACTED_IN]->(m), (b)-[:ACTED_IN]->(m)` find the same thing.
3. **Keanu is his own co-actor.** Look at the result: he shows up three times in his own list, once for each film he's in. The pattern `(keanu)-[:ACTED_IN]->(m)<-[:ACTED_IN]-(other)` doesn't prevent `other` from matching `keanu`. You'd add `WHERE other.name <> 'Keanu Reeves'` to exclude him. We'll get fluent with `WHERE` in NB2.
4. **There's no LLM in any of this.** Graph databases are useful by themselves. The LLM in NB3 is for *translating natural language into queries like this one*. The retrieval itself is plain Cypher.

That's also why a real schema (with declared node and relationship types) matters so much: the LLM gets to see the structure when it's deciding what query to write.

## Recap

- A graph database stores nodes, relationships (directed), and properties.
- You declare a schema with `CREATE NODE TABLE` and `CREATE REL TABLE`.
- You insert with `CREATE (:Label {...})` and `MATCH ... CREATE (a)-[:REL]->(b)`.
- You query with `MATCH ... RETURN`. The pattern is the query.
- Two-hop traversal (co-actors) is one line of Cypher.

[NB2](./02_querying_the_graph.ipynb) drops the toy schema and loads the full movie graph. The Cypher gets longer; the moves get more interesting.

## Cleanup

The throwaway demo DB is just a single file. Delete it any time, or leave it for re-running the notebook.

In [6]:
removed = []
for path in [DB_PATH, DB_PATH.with_suffix(DB_PATH.suffix + ".wal")]:
    if path.exists():
        path.unlink()
        removed.append(str(path))

if removed:
    print(f"Removed: {', '.join(removed)}")
else:
    print(f"{DB_PATH} already gone.")

Removed: data\intro_demo.kuzu, data\intro_demo.kuzu.wal
